# 03 (Block 2) — Multi-Agent Topologies: Sequential, Parallel & Bounded Loop
**Official Reference Pattern**: Google 5-Day AI Agents Intensive (`day-1b-agent-architectures.ipynb`)
**Antigravity IDE Usage**: **PRIMARY REFERENCE FOR LAB 1 (Block 2)**.

## Lab 1 Cross-Porting Instructions
During **Lab 1**, you will use Antigravity Agent Manager to port the Sequential, Parallel, and bounded LoopAgent workflow below into TypeScript. Lab instructions are released live in `labs/block-2/`.

## Key Slow Engineering Invariants Demonstrated
* **`output_key` State Handoffs**: Agents write structured artifacts into `session.state[output_key]` (`plan_artifact`, `draft_code`, `verification_report`).
* **Bounded `LoopAgent` (`max_iterations=2`)**: Never deploy ungated 24/7 loops. Pair `ExecutorAgent` with an independent `VerifierAgent` that calls `exit_loop` only when deterministic checks pass.


In [2]:
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.tools import ToolContext

def exit_verification_loop(tool_context: ToolContext) -> dict:
    """Called by the independent VerifierAgent when all checks pass (Jidoka stop/approve)."""
    tool_context.actions.escalate = True
    return {"status": "APPROVED", "message": "All deterministic checks passed. Exiting loop."}

# 1. Parallel Discovery Fan-Out (Isolated Research Sandboxes)
spec_researcher = Agent(
    name="SpecResearcher",
    model="gemini-3.5-flash-lite",
    instruction="Extract strict input/output type constraints for the requested feature.",
    output_key="spec_constraints",
)

security_researcher = Agent(
    name="SecurityResearcher",
    model="gemini-3.5-flash-lite",
    instruction="Identify Principle 10 (Every Input is a Stranger) validation boundaries.",
    output_key="security_constraints",
)

parallel_discovery = ParallelAgent(
    name="ParallelDiscoveryPhase",
    sub_agents=[spec_researcher, security_researcher],
)

# 2. Bounded Execute + Verify Loop (Jidoka for Agents)
executor_agent = Agent(
    name="ExecutorAgent",
    model="gemini-3.8-flash",
    instruction=(
        "Implement the module using {spec_constraints} and {security_constraints}. "
        "If {verification_report} exists, fix all flagged issues."
    ),
    output_key="draft_artifact",
)

verifier_agent = Agent(
    name="IndependentVerifierAgent",
    model="gemini-4-argon",
    instruction=(
        "Slow Engineering Principle 04: Never let an agent mark its own homework. "
        "Inspect {draft_artifact} against {spec_constraints} and {security_constraints}. "
        "If all checks pass, call exit_verification_loop(). Otherwise output a defect list."
    ),
    tools=[exit_verification_loop],
    output_key="verification_report",
)

bounded_verify_loop = LoopAgent(
    name="PlanExecuteVerifyLoop",
    sub_agents=[executor_agent, verifier_agent],
    max_iterations=2,  # Slow Engineering Principle 06: Fail Soft (strict iteration cap)
)

# 3. Root Sequential Pipeline: Parallel Discovery -> Bounded Execute/Verify Loop -> Decision Card
decision_card_emitter = Agent(
    name="DecisionCardEmitter",
    model="gemini-3.8-flash",
    instruction=(
        "Summarize {draft_artifact} and {verification_report} into a concise Morning Decision Card: "
        "whatWasDone, evidenceSummary, checksPassed, and the one questionForHuman."
    ),
    output_key="decision_card",
)

root_team = SequentialAgent(
    name="AxiomSlowEngineeringTeam",
    sub_agents=[parallel_discovery, bounded_verify_loop, decision_card_emitter],
)

print("Constructed Root Topology:", root_team.name)
for sub in root_team.sub_agents:
    print(f"  -> Stage: {sub.name} ({type(sub).__name__})")


Constructed Root Topology: AxiomSlowEngineeringTeam
  -> Stage: ParallelDiscoveryPhase (ParallelAgent)
  -> Stage: PlanExecuteVerifyLoop (LoopAgent)
  -> Stage: DecisionCardEmitter (Agent)


In [3]:
# Simulated Execution Trace & Resulting Decision Card Artifact
sample_decision_card = {
    "jobId": "job-2026-1110-01",
    "whatWasDone": "Implemented MCP tool schema validator with strict Zod input boundary.",
    "evidenceSummary": "Passed 2/2 VerifierAgent checks (Iteration 1 flagged missing max-length cap; Iteration 2 passed).",
    "checksPassed": True,
    "questionForHuman": "Approve merging schema validator into Lab 2, or hold for manual review?"
}
import json
print(json.dumps(sample_decision_card, indent=2))


{
  "jobId": "job-2026-1110-01",
  "whatWasDone": "Implemented MCP tool schema validator with strict Zod input boundary.",
  "evidenceSummary": "Passed 2/2 VerifierAgent checks (Iteration 1 flagged missing max-length cap; Iteration 2 passed).",
  "checksPassed": true,
  "questionForHuman": "Approve merging schema validator into Lab 2, or hold for manual review?"
}
